# Dependencies

Install the required dependencies and libraries

In [ ]:
%pip install langchain langchain_core langchain-huggingface ipywidgets python-dotenv gqlalchemy langchain-memgraph langgraph langgraph-cli[inmem] langchain-anthropic langchain-openai jinja2 pandas scikit-learn scipy statsmodels numpy matplotlib

# Environment Variables and Constants

Load the configuration options from the environment

In [ ]:
from dotenv import load_dotenv

load_dotenv(dotenv_path="./.env")

The system prompt instructs the AI on how to approach the vulnerability assessment, what criteria to follow for analysis and how to report the output.
It also defines the constraints.

In [ ]:
from langchain.messages import SystemMessage

SYSTEM_MSG = SystemMessage("""
You are an AI Cybersecurity Risk Analyst operating inside an enterprise
vulnerability prioritization system.

Your purpose is to assess the REAL-WORLD RISK of a vulnerability or CVE
using enterprise context stored in a knowledge graph.

You do NOT rely only on CVSS severity.
You MUST perform context-aware reasoning using the available graph data.

--------------------------------------------------
CORE OBJECTIVE
--------------------------------------------------

Given a vulnerability identifier (CVE or Vulnerabiliy name), you must:

1. Investigate the vulnerability using the knowledge graph.
2. Collect all relevant contextual information.
3. Evaluate risk using the Risk Evaluation Framework explained below.
4. Produce a justified risk assessment with reasoning.

You MUST query the graph before making conclusions.

--------------------------------------------------
KNOWLEDGE GRAPH SEMANTICS
--------------------------------------------------

The graph models an enterprise IT environment:

Nodes:
- Vulnerability
- Asset
- Service
- ThreatSignal
- Package                           

Relationships:
- (Vulnerability)-[:AFFECTS]->(Asset)
- (Asset)-[:HOSTS]->(Service)
- (Service)-[:DEPENDS_ON]->(Service)
- (Asset)-[:DEPENDS_ON]->(Package)
- (Vulnerability)-[:HAS]->(ThreatSignal)

You must traverse these relationships to understand impact.

--------------------------------------------------
RISK EVALUATION FRAMEWORK
--------------------------------------------------

The framework considers each vulnerability instance within its comprehensive enterprise context.
Assess risk using FIVE contextual dimensions:

1. Vulnerability Severity $S_{cvss}(v)$
   - CVSS or intrinsic vulnerability properties.

2. Deployment Exposure $S_{exp}(v)$
   - Internet exposure
   - Environment (prod > staging > dev)
   - Data classification

3. Business Criticality $S_{crit}(v)$
   - Service criticality
   - Revenue impact
   - PII handling

4. Exploit Likelihood $S_{exploit}(v)$
   - EPSS score
   - KEV listing
   - Public exploit availability

5. Blast Radius $S_{blast}(v)$
   - Service dependencies
   - Downstream affected services
   - Shared components

You MUST gather evidence for each dimension from the graph.
Each componenet MUST be normalized to a score of 0-10 and combined using the formula:
$$S_{priority}(v) = S_{cvss}(v) + S_{exp}(v) + S_{crit}(v) + S_{exploit}(v) + S_{blast}(v)$$

--------------------------------------------------
REASONING PROCESS (MANDATORY)
--------------------------------------------------

Always follow this workflow:

Step 1 — Retrieve knowledge graph schema.                           
Step 2 — Identify vulnerability node.
Step 3 — Retrieve affected assets.
Step 4 — Determine hosted services.
Step 5— Analyze service and asset criticality and exposure.
Step 6 — Retrieve threat intelligence signals.
Step 7 — Analyze dependency graph to estimate blast radius.
Step 8 — Integrate all signals into a contextual risk judgment.

Do NOT skip steps.

If information is missing, state assumptions explicitly.

--------------------------------------------------
TOOL USAGE RULES
--------------------------------------------------

- Use graph tools to query data whenever context is required.
- Prefer multiple focused queries over one large query.
- Never fabricate graph data.
- Never assume relationships without querying.

--------------------------------------------------
OUTPUT REQUIREMENTS
--------------------------------------------------

Return a structured risk assessment containing score breakdown for each dimension

Your reasoning must reference discovered context.
                           
--------------------------------------------------
IMPORTANT CONSTRAINTS
--------------------------------------------------

You are an analytical system, NOT a chatbot.

Do NOT:
- provide generic vulnerability advice
- rely only on CVSS
- skip graph exploration
- hallucinate enterprise context

Your conclusions must be grounded in graph evidence.

Always think like a security analyst investigating enterprise risk.
"""
                           )

# Data Loaders

Defines the methods used for loading and ingesting the data

Specify databse connection parameters

In [ ]:
import os

from gqlalchemy import Memgraph

# Connect to memgraph
url = os.getenv("MEMGRAPH_URL")
port = int(os.getenv("MEMGRAPH_PORT"))
memgraph = Memgraph(url, port)

Clear the existing database. This is required for first time setup so that there are no duplicate nodes and relationships

In [ ]:
# clear existing data
memgraph.drop_database()

In [ ]:
def ingest_data_to_graph():
    """
    Ingests data into memgraph
    Loads the vulnerability dataset from json files into
    the graph database by running cypher queries. The data files need to be accessible
    by the memgraph or MAGE service.
    """

    # Connect to memgraph
    url = os.getenv("MEMGRAPH_URL")
    port = int(os.getenv("MEMGRAPH_PORT"))
    memgraph = Memgraph(url, port)

    # clear existing data
    memgraph.drop_database()

    # read query file content
    with open("query.cql", 'r') as f:
        query = f.read()

    # execute the query to ingest data
    memgraph.execute(query)

Ingest the enterprise json data to the graph database by calling the ingestion method

In [ ]:
ingest_data_to_graph()

# Data Formatting

Methods used for formatting and converting the data

Define the data model for the response the AI should return

In [ ]:
from pydantic import BaseModel


class RiskAssessment(BaseModel):
    vulnerability_id: str
    contextual_risk_score: float
    contextual_priority_score: float
    risk_level: str
    reasoning: str
    key_risk_drivers: list[str]
    affected_services: list[str]
    blast_radius_summary: str
    confidence_level: float
    priority: str

# Utility Methods

Utility methods are used to load models and setup database connections

Graph database utility methods. These are used to create database connection instnaces and databse tools

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_huggingface import ChatHuggingFace
from langchain_memgraph import MemgraphToolkit
from langchain_memgraph.graphs.memgraph import MemgraphLangChain
from langchain_openai import ChatOpenAI


def connect_to_memgraph() -> MemgraphLangChain:
    """
    Connect to memgraph database instance

    Returns:
        `MemgraphLangChain` instance
    """

    db = MemgraphLangChain(
        url=f"{os.getenv("MEMGRAPH_DRIVER")}://{os.getenv("MEMGRAPH_URL")}:{os.getenv("MEMGRAPH_PORT")}",
        username='',
        password=''
    )
    return db


def get_memgraph_tools(db: MemgraphLangChain, model: ChatHuggingFace | ChatAnthropic | ChatOpenAI) -> List:
    """
    Retrieves memgraph langchain tools from memgraph toolkit

    Parameters:
        db (MemgraphLangChain): Database instance object
        model (ChatHuggingFace): LLM model object

    Returns:
        A list of memgraph tools
    """

    toolkit = MemgraphToolkit(
        db=db,
        llm=model
    )
    tools = toolkit.get_tools()
    return tools

Methods used for loading the models from different providors

In [ ]:
import os

from langchain_anthropic import ChatAnthropic
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_openai import ChatOpenAI


def load_model_from_hf(repo_id: str) -> ChatHuggingFace:
    """
    Loads a ChatHuggingFace model from Hugging Face using the Novita provider.

    This function initializes a HuggingFaceEndpoint with the specified repository ID
    and configures it as a ChatHuggingFace model for conversational interactions.
    The model is loaded via Novita's inference service.

    Args:
        repo_id (str): The Hugging Face repository ID of the model to load
                      (e.g., 'deepseek-ai/DeepSeek-V3.2').

    Returns:
        ChatHuggingFace: A configured ChatHuggingFace instance ready for chat interactions.
    """

    llm = HuggingFaceEndpoint(
        repo_id=repo_id,
        provider="novita",
        huggingfacehub_api_token=os.getenv("HUGGINGFACEHUB_API_TOKEN")
    )

    model = ChatHuggingFace(llm=llm)
    return model


def load_novita_model(model_name: str) -> ChatOpenAI:
    """
    Loads a ChatOpenAI model via the Novita API.

    This function creates a ChatOpenAI instance configured to use Novita's API endpoint,
    allowing access to OpenAI-compatible models through Novita's service. The model
    is set up with structured responses, a maximum token limit, and moderate temperature.

    Args:
        model_name (str): The name of the model to load via Novita
                         (e.g., 'deepseek/deepseek-v3.2').

    Returns:
        ChatOpenAI: A configured ChatOpenAI instance for chat interactions.
    """

    model = ChatOpenAI(
        model=model_name,
        api_key=os.getenv("NOVITA_API_KEY"),
        base_url="https://api.novita.ai/openai",
        use_responses_api=True,
        max_tokens=4000,
        temperature=0.5
    )

    return model


def load_anthropic_model(model_name: str) -> ChatAnthropic:
    """
    Loads a ChatAnthropic model for conversational AI interactions.

    This function initializes a ChatAnthropic instance with the specified model name,
    configured with high effort, adaptive thinking, maximum token limits, and moderate
    temperature for optimal performance in vulnerability assessment tasks.

    Args:
        model_name (str): The name of the Anthropic model to load
                         (e.g., 'claude-opus-4-6').

    Returns:
        ChatAnthropic: A configured ChatAnthropic instance ready for chat interactions.
    """

    model = ChatAnthropic(
        model=model_name,
        anthropic_api_key=os.getenv("ANTHROPIC_API_KEY"),
        effort="high",
        thinking={"type": "adaptive"},
        max_tokens=4000,
        temperature=0.5,
    )

    return model


def load_openai_model(model_name: str) -> ChatOpenAI:
    """
    Loads a ChatOpenAI model for conversational AI interactions.

    This function creates a ChatOpenAI instance configured with the OpenAI API,
    using structured responses, medium reasoning effort, maximum token limits,
    and moderate temperature for balanced performance.

    Args:
        model_name (str): The name of the OpenAI model to load
                         (e.g., 'gpt-5.1').

    Returns:
        ChatOpenAI: A configured ChatOpenAI instance for chat interactions.
    """
    
    model = ChatOpenAI(
        model=model_name,
        api_key=os.getenv("OPENAI_API_KEY"),
        use_responses_api=True,
        reasoning_effort="medium",
        max_tokens=4000,
        temperature=0.5
    )

    return model

Memgraph query methods.

These are used to load different vulnerability data from the graph database

In [ ]:
query_vuln = """
MATCH (v:Vulnerability {{name: "{cve}"}})
RETURN properties(v) AS vuln
"""

query_threat_signal = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[r:HAS]->(ts:ThreatSignal)
RETURN properties(ts) AS threatsignal
"""

query_affected_assets = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[r:AFFECTS]->(a:Asset)
RETURN properties(a) AS asset
"""

query_affected_services = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[r:AFFECTS]->(a:Asset)-[:HOSTS]->(s:Service)
RETURN properties(s) AS service
"""

query_packages = """
MATCH (v:Vulnerability {{name: "{cve}"}})-[:AFFECTS]->(a:Asset)-[:DEPENDS_ON]->(p:Package)
WITH collect(DISTINCT p.name) as packages
RETURN packages
"""

query_service_dependencies = """

"""

In [ ]:
def get_query_vuln(cve_id):
    result = memgraph.execute_and_fetch(query_vuln.format(cve=cve_id))
    return list(result)[0]['vuln']

def get_query_threat_signal(cve_id):
    result = memgraph.execute_and_fetch(query_threat_signal.format(cve=cve_id))
    return list(result)[0]['threatsignal']

def get_query_affected_assets(cve_id):
    result = memgraph.execute_and_fetch(query_affected_assets.format(cve=cve_id))
    assets = [record["asset"] for record in list(result)]
    return assets

def get_query_affected_services(cve_id):
    result = memgraph.execute_and_fetch(query_affected_services.format(cve=cve_id))
    services = [record["service"] for record in list(result)]
    return services

def get_query_packages(cve_id):
    result = memgraph.execute_and_fetch(query_packages.format(cve=cve_id))
    return list(result)[0]['packages']

In [ ]:
get_query_affected_assets("CVE-2025-67221")

## Middleware

Agent middleware hook definitions. 

Middleware to retry databse cypher queries.

There are some instance where open access models (deepseek-v3.2) use older query schema which results in errors. The middleware ensures the query errors are returned as `ToolMessage` to the model to retry the queries instead of breaking the workflow.

In [ ]:
from typing import Any, Callable

from langchain.agents.middleware import (AgentMiddleware, ModelRequest,
                                         ModelResponse)
from langchain.messages import ToolMessage
from neo4j.exceptions import ClientError


class RetryCypherMiddleware(AgentMiddleware):
    """
    Middleware for handling Cypher query errors in agent tool calls.

    This middleware intercepts tool calls that execute Cypher queries against the graph database.
    When a Neo4j ClientError occurs (such as due to outdated query schemas from certain models),
    it catches the exception and returns a user-friendly ToolMessage instead of propagating
    the raw database error, allowing the agent to retry the query with corrected syntax.
    """

    def wrap_tool_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse[Any]:
        """
        Handles synchronous tool calls by wrapping them with error handling for Cypher queries.

        This method executes the tool call handler and catches any ClientError exceptions
        that may arise from invalid or outdated Cypher query syntax. Instead of failing
        the entire agent workflow, it returns a standardized error message as a ToolMessage,
        preserving the original tool call ID to enable retry mechanisms.

        Args:
            request (ModelRequest): The incoming model request containing tool call context
                                   and parameters.
            handler (Callable[[ModelRequest], ModelResponse]): The synchronous function
                                                               responsible for executing the tool call.

        Returns:
            ModelResponse: Either the successful response from the handler or a ToolMessage
                           containing a user-friendly error message on database query failure.
        """

        try:
            return handler(request)

        except ClientError as err:
            return ToolMessage(
                content=f"Tool error: Please check your input and try again. ({str(err)})",
                tool_call_id=request.tool_call["id"]
            )

    async def awrap_tool_call(self, request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse[Any]:
        """
        Handles asynchronous tool calls by wrapping them with error handling for Cypher queries.

        This async method executes the tool call handler and catches any ClientError exceptions
        that may arise from invalid or outdated Cypher query syntax. Instead of failing
        the entire agent workflow, it returns a standardized error message as a ToolMessage,
        preserving the original tool call ID to enable retry mechanisms.

        Args:
            request (ModelRequest): The incoming model request containing tool call context
                                   and parameters.
            handler (Callable[[ModelRequest], ModelResponse]): The asynchronous function
                                                               responsible for executing the tool call.

        Returns:
            ModelResponse: Either the successful response from the handler or a ToolMessage
                           containing a user-friendly error message on database query failure.
        """
        
        try:
            return await handler(request)

        except ClientError as err:
            return ToolMessage(
                content=f"Tool error: Please check your input and try again. ({str(err)})",
                tool_call_id=request.tool_call["id"]
            )

Middleware hook that checks for truncated model output.

When using HuggingFace Inference API, output tokens are capped at 512 which prematurely cuts of the model response. This hook ensures model is prompted again to continue generating the output, which utlimately returns the complete output.

Do note that this behaviour has only been observed when using models loading from `HuggingFaceEndpoint`. For other providors, this middleware hook is not required.

In [ ]:
from typing import Any

from langchain.agents.middleware import AgentState, after_model, hook_config
from langchain.messages import HumanMessage
from langgraph.runtime import Runtime


@after_model
@hook_config(can_jump_to=["model"])
def check_truncated_output(state: AgentState, runtime: Runtime) -> dict[Any, Any] | None:
    """
    Middleware hook that detects truncated model responses and triggers continuation.

    This function is executed after each model response in the agent workflow. It checks
    if the model's output was truncated due to token limits (indicated by finish_reason="length").
    When truncation is detected, it returns instructions to jump back to the model node
    with a continuation prompt, allowing the model to generate the remaining output.

    This is particularly useful for HuggingFace Inference API models that have strict
    token caps (e.g., 512 tokens), ensuring complete responses are obtained through
    iterative continuation rather than incomplete outputs.

    Args:
        state (AgentState): The current agent state containing message history and
                           workflow context. Used to access the most recent model response.
        runtime (Runtime): The LangGraph runtime object providing execution context
                          and control flow capabilities.

    Returns:
        dict[Any] | None: A dictionary with "messages" and "jump_to" keys to trigger
                         workflow continuation if output was truncated, or None if
                         the response is complete. The messages contain a continuation
                         prompt asking the model to resume from where it left off.
    """
    
    last_message = state["messages"][-1]

    if last_message.response_metadata["finish_reason"] == "length":
        return {
            "messages": [HumanMessage("Continue from precisely where you left off.")],
            "jump_to": "model"
        }
    return None

# Workflow Execution

Loads the configurations and starts the pipeline execution.

Create the graph database instance

In [ ]:
# Get memgraph database instance
db = connect_to_memgraph()

Load the required models

In [ ]:
# load the models
# hf_model = load_model_from_hf("deepseek-ai/DeepSeek-V3.2")
novita_model = load_novita_model("deepseek/deepseek-v3.2")
anthropic_model = load_anthropic_model("claude-opus-4-6")
openai_model = load_openai_model("gpt-5.1")

Create the agents. Three agents are created covering two categories
* Open Access Models
* Frontier Models

In [ ]:
from langchain.agents import create_agent
from langchain.agents.structured_output import ProviderStrategy

# instantiate hugging face agent
# hf_agent = create_agent(
#     hf_model,
#     get_memgraph_tools(db, hf_model),
#     system_prompt=SYSTEM_MSG,
#     middleware=[check_truncated_output, retry_cypher]
# ).with_config({"run_name": "HuggingFaceAgent"})

# instantiate novita agent
novita_agent = create_agent(
    novita_model,
    get_memgraph_tools(db, novita_model),
    system_prompt=SYSTEM_MSG,
    # response_format=RiskAssessment,
    middleware=[RetryCypherMiddleware()]
).with_config({
    "run_name": "NovitaAgent",
    "tags": ["novita", "deepseek"]
})

# instantiate anthropic agent
anthropic_agent = create_agent(
    anthropic_model,
    get_memgraph_tools(db, anthropic_model),
    system_prompt=SYSTEM_MSG,
    # response_format=ProviderStrategy(RiskAssessment)
).with_config({
    "run_name": "AnthropicAgent",
    "tags": ["anthropic", "opus-4.6"]
    })

# instantiate openai agent
openai_agent = create_agent(
    openai_model,
    get_memgraph_tools(db, openai_model),
    system_prompt=SYSTEM_MSG
    # response_format=ProviderStrategy(RiskAssessment)
).with_config({
    "run_name": "OpenAIAgent",
    "tags": ["openai", "gpt-5"]
    })

Display the agent stategraphs

In [ ]:
from IPython.display import Image, display

# display the stategraphs
display(Image(novita_agent.get_graph().draw_mermaid_png()))
display(Image(anthropic_agent.get_graph().draw_mermaid_png()))
display(Image(openai_agent.get_graph().draw_mermaid_png()))

Wrap the agent objects inside a `RunnableParallel` instance for concurrent execution of user queries

In [ ]:
from langchain_core.runnables import RunnableParallel

parallel_agents = RunnableParallel(
    novita=novita_agent,
    anthropic=anthropic_agent,
    openai=openai_agent,
)

Define input messages

In [ ]:
from langchain.messages import HumanMessage

user_message = {
    "messages": [
        HumanMessage("What is the risk posed by CVE-2025-67725?")
    ]
}

user_message1 = {
    "messages": [
        HumanMessage("Tell me a joke.")
    ]
}

user_message2 = {
    "messages": [
        HumanMessage("When did the Roman Empire Fall")
    ]
}

### Run the agents

Run a single agent with a user query

In [ ]:
result = novita_agent.invoke(user_message)

Run a single user query against multiple agents concurrently by executing on the `RunnableParallel`

In [ ]:
results = parallel_agents.invoke(user_message)

Run the user request in batches using `abatch` method. This executes agentic workflow for each user query concurrently 

In [ ]:
results = parallel_agents.abatch(
    inputs=[user_message1, user_message2],
    config={"max_concurrency": 7}
)

In [ ]:
structured_anthropic_model = anthropic_model.with_structured_output(RiskAssessment)

In [ ]:
risk_assessment = structured_anthropic_model.invoke(
    f"Given the vulnerability analysis, strcuture it according to the RiskAssessment schema: \n\n {final_text}"
)

# Evaluation

## Ground Truth

The first step to evaluation and analysis is to create a curated ground truth dataset.

Ground truth is defined as expert labelled risk assessment of vulnerability instances. For this, first assemble a stratified sample of vulnerabilites.

### Stratification

The sample set of vulnerabilities is created over stratified axes based on:

- CVSS severity band (Low / Medium / High / Critical)
- Network exposure (internet-facing / partner / internal)
- Asset tier (mission-critical or critical / important or supporting)



First we extract all the vulnerabilities data with context into a csv and load it into a pandas dataframe.

Run the `queries/all_vuln_with_context.cql` and extract the data into `results/all_vuln_with_context.csv`

In [ ]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd

# Define path where result files are stored/will be stored
RESULTS_DIR = Path.cwd() / "results"

# load all vulnerabilities
vulns = pd.read_csv(RESULTS_DIR / "all_vulns_with_context.csv")

# parse the string of lists
vulns["exposures"] = vulns["exposures"].apply(ast.literal_eval)
vulns["tiers"]     = vulns["tiers"].apply(ast.literal_eval)

# collapse the exposure and tier lists into a single "worst case" value for each 

EXPOSURE_RANK = {"internet-facing": 3, "partner-network": 2, "internal": 1}
TIER_RANK     = {"mission-critical": 4, "critical": 3, "important": 2, "supporting": 1}

def worst_exposure(exposures):
    return max(exposures, key=lambda e: EXPOSURE_RANK.get(e, 0))

def worst_tier(tiers):
    return max(tiers, key=lambda t: TIER_RANK.get(t, 0))

vulns["max_exposure"] = vulns["exposures"].apply(worst_exposure)
vulns["max_tier"]     = vulns["tiers"].apply(worst_tier)

Now that the data is in shape, we can proceed with defining the stratas.

The strats are defined on the 3 axis defined earlier. For each possible axis, we define some values:

- CVSS - 4 possible values
- Exposure - 3 possible values -> Collapse into 2
- Criticality Tier - 4 possible values -> Collapse into two

This bucketing gives us 4x2x2=16 cells. A distribution is created as sanity check before proceeding with the sampling

In [ ]:
# bucket into stratas

def cvss_band(c):
    if c < 4.0:  return "Low"
    if c < 7.0:  return "Medium"
    if c < 9.0:  return "High"
    return "Critical"

def exposure_band(e):
    return "external" if e in ("internet-facing", "partner-network") else "internal"

def tier_band(t):
    return "high" if t in ("mission-critical", "critical") else "low"

vulns["cvss_band"]     = vulns["cvss"].apply(cvss_band)
vulns["exposure_band"] = vulns["max_exposure"].apply(exposure_band)
vulns["tier_band"]     = vulns["max_tier"].apply(tier_band)

# sanity check of distribution before sampling
distribution = vulns.groupby(
    ["cvss_band","exposure_band","tier_band"]
).size().reset_index(name="count")
distribution.to_csv(RESULTS_DIR / "distribution.csv", index=False)
print(distribution)

  cvss_band exposure_band tier_band  count
0  Critical      external      high     20
1      High      external      high    125
2       Low      external      high     14
3    Medium      external      high    126


In [6]:
print(vulns)

             cve_id  cvss                                        description  \
0     CVE-2026-4539   4.8       Regular Expression Denial of Service (ReDoS)   
1     CVE-2026-4438   4.0                                      CVE-2026-4438   
2     CVE-2026-4105   6.7                            Improper Access Control   
3    CVE-2026-34073   6.3                    Improper Certificate Validation   
4    CVE-2026-34043   8.2  Allocation of Resources Without Limits or Thro...   
..              ...   ...                                                ...   
280  CVE-2026-26278   8.7                               XML Entity Expansion   
281  CVE-2026-26996   8.7       Regular Expression Denial of Service (ReDoS)   
282  CVE-2026-27205   6.5      Use of Cache Containing Sensitive Information   
283  CVE-2023-31438   5.3       Improper Validation of Integrity Check Value   
284  CVE-2026-27699   9.8                                Directory Traversal   

        epss    kev  public_exploit  \


Now we sample the vulnerabilities from the stratum. We sample instances into each cell with oversampling in disagreement prone cells.

Visullay, this will look something like:

| Cell | Reasoning | n |
| :--: | :--: | :--: |
| High/Critical CVSS + internal + low-tier | CVSS over-prioritizes | 6 |
| Low/Medium CVSS + external + high-tier | CVSS under-prioritizes | 8 |
| Everything else | Baseline | 4 |

In [ ]:
# defining per-cell sample sizes

def target_n(cvss, expo, tier):
    # oversampling in disagreement prone cells
    if cvss in ("High","Critical") and expo == "internal" and tier == "low":
        return 6
    if cvss in ("Low","Medium") and expo == "external" and tier == "high":
        return 8
    # baseline
    return 4

# Sample from each cell

rng = np.random.default_rng(seed=42)  # hardcoded seed for reproducibility
samples = []
for (cvss, expo, tier), group in vulns.groupby(["cvss_band","exposure_band","tier_band"]):
    want = target_n(cvss, expo, tier)
    got  = min(want, len(group))
    if got < want:
        print(f"WARN: cell ({cvss}, {expo}, {tier}) has only {len(group)} vulns, wanted {want}")
    samples.append(group.sample(n=got, random_state=rng.integers(1e9)))

sample_df = pd.concat(samples).reset_index(drop=True)
print(f"Total sample size: {len(sample_df)}")

# save labelling samples
sample_df.to_csv(RESULTS_DIR / "labeling_sample.csv", index=False)

# save the cell-count table for Chapter 3
cell_counts = sample_df.groupby(
    ["cvss_band","exposure_band","tier_band"]
).size().reset_index(name="n_sampled")
cell_counts.to_csv("datasets/extracts/stratification_breakdown.csv", index=False)
print(cell_counts)

Total sample size: 24
  cvss_band exposure_band tier_band  n_sampled
0  Critical      external      high          4
1      High      external      high          4
2       Low      external      high          8
3    Medium      external      high          8


### Labelling Packets

Now that we have a stratified sample set of vulnerabilities, the next step is to generate labelling packats for each vulnerability instance.

The labelling packets are what will be distributed to the human experts to collect their expert labels that will later on form the ground truth dataset.

Define the markdown template for the labelling packets and corresponding methods to generate each packet

In [ ]:
from jinja2 import Template

TEMPLATE = Template("""
# Vulnerability Labeling Packet — Item {{ item_num }} of {{ total }}

## Vulnerability
- **CVE:** {{ cve }}
- **Description:** {{ description }}
- **CVSS:** {{ cvss }}

## Threat Signals
- EPSS: {{ epss }}
- KEV Listed: {{ kev }}
- Public Exploit Available: {{ public_exploit }}

## Affected Assets ({{ assets|length }})
| Hostname | Env | Tier | Exposure | Data Class |
|---|---|---|---|---|
{% for a in assets -%}
| {{ a.hostname }} | {{ a.environment }} | {{ a.tier }} | {{ a.network_exposure }} | {{ a.data_classification }} |
{% endfor %}

## Hosted Services
{% for s in services -%}
- **{{ s.name }}** ({{ s.criticality }}, revenue={{ s.revenue_impact }}, PII={{ s.handles_pii }})
  {{ s.description }}
{% endfor %}

## Package Dependencies
{{ packages|join(", ") }}



---

## Labeling Form

**Priority tier:**  [ ] P1   [ ] P2   [ ] P3   [ ] P4

**Fix urgency:**  [ ] must-fix-now   [ ] near-term   [ ] scheduled   [ ] defer

**Confidence (1–5):**  _____

**Rationale (2–4 sentences):**
_______________________________________________
_______________________________________________

**Time spent (minutes):**  _____

**Additional context you would have wanted:**
_______________________________________________
""")

In [ ]:
def generate_packet(cve_id, item_num, total):
    # Query Memgraph for everything about this CVE
    vuln = get_query_vuln(cve_id)
    assets = get_query_affected_assets(cve_id)
    services = get_query_affected_services(cve_id)
    packages = get_query_packages(cve_id)
    # service_deps = query_service_dependencies(cve_id)
    threat = get_query_threat_signal(cve_id)
    
    return TEMPLATE.render(
        item_num=item_num, total=total,
        cve=cve_id, description=vuln["description"], cvss=vuln["cvss"],
        epss=threat["epss"], kev=threat["kevListed"], 
        public_exploit=threat["publicExploit"],
        assets=assets, services=services, 
        packages=packages, #service_deps=service_deps,
    )

Now we generate labelling packates for each vulnerability instance from the stratified sample

In [ ]:
# generate labelling packets from stratified samples
sample = pd.read_csv(RESULTS_DIR / "labeling_sample.csv")
for i, row in enumerate(sample.itertuples(), 1):
    md = generate_packet(row.cve_id, i, len(sample))
    with open(RESULTS_DIR / f"packets/packet_{i:02d}_{row.cve_id}.md", "w") as f:
        f.write(md)

Each vulnerability packet now exists in markdown. For ease of comprehension and distribution, we create pdfs for each labelling packet

In [ ]:
# %% [markdown]
# # Convert Labeling Packets (Markdown → PDF)
# Prerequisites: `sudo apt install pandoc wkhtmltopdf`

# %%
from pathlib import Path
import subprocess

PACKETS_DIR = Path("packets")
OUTPUT_DIR = PACKETS_DIR / "pdf"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# %%
# CSS for clean, professional labeling packets
PACKET_CSS = """
body {
    font-family: "Segoe UI", "Helvetica Neue", Arial, sans-serif;
    font-size: 11pt;
    line-height: 1.5;
    color: #1a1a1a;
    max-width: 100%;
    margin: 0;
    padding: 0;
}
h1 {
    font-size: 16pt;
    border-bottom: 2px solid #2c3e50;
    padding-bottom: 6px;
    margin-top: 0;
}
h2 {
    font-size: 13pt;
    color: #2c3e50;
    border-bottom: 1px solid #bdc3c7;
    padding-bottom: 4px;
    margin-top: 20px;
}
table {
    border-collapse: collapse;
    width: 100%;
    margin: 10px 0;
    font-size: 10pt;
}
th {
    background-color: #2c3e50;
    color: white;
    padding: 6px 10px;
    text-align: left;
}
td {
    border: 1px solid #ddd;
    padding: 5px 10px;
}
tr:nth-child(even) {
    background-color: #f8f9fa;
}
hr {
    border: none;
    border-top: 2px solid #2c3e50;
    margin: 25px 0;
}
code {
    background-color: #f4f4f4;
    padding: 1px 4px;
    border-radius: 3px;
    font-size: 10pt;
}
strong {
    color: #2c3e50;
}
ul {
    margin: 5px 0;
}
"""

css_path = PACKETS_DIR / "_packet_style.css"
css_path.write_text(PACKET_CSS)

# %%
# Convert all packets
md_files = sorted(PACKETS_DIR.glob("packet_*.md"))
print(f"Found {len(md_files)} packets\n")

for md in md_files:
    pdf_path = OUTPUT_DIR / (md.stem + ".pdf")
    
    cmd = [
        "pandoc", str(md), "-o", str(pdf_path),
        "--pdf-engine=wkhtmltopdf",
        "--css", str(css_path),
        "--metadata", f"title={md.stem}",
        "--pdf-engine-opt", "--margin-top",    "--pdf-engine-opt", "20mm",
        "--pdf-engine-opt", "--margin-bottom", "--pdf-engine-opt", "20mm",
        "--pdf-engine-opt", "--margin-left",   "--pdf-engine-opt", "20mm",
        "--pdf-engine-opt", "--margin-right",  "--pdf-engine-opt", "20mm",
        "--pdf-engine-opt", "--enable-local-file-access",
    ]
    
    result = subprocess.run(cmd, capture_output=True, text=True)
    
    if result.returncode == 0:
        print(f"OK    {pdf_path}  ({pdf_path.stat().st_size / 1024:.0f} KB)")
    else:
        print(f"FAIL  {md.name}\n      {result.stderr[:200]}")

# clean up
css_path.unlink(missing_ok=True)
print(f"\nDone. PDFs in {OUTPUT_DIR}/")

## Analysis

Perform statistical analysis to answer the research questions.

First setup the ground truth and the baselines that will be used for comparisons later on

In [ ]:
import pandas as pd

# load the results table
df = pd.read_csv("results/master_results.csv")

# the ground truth.
# this is what every method is compared against.
y_true = df["Consensus_tier"]

# CVSS-only baseline: map CVSS to tiers using standard thresholds
def cvss_to_tier(cvss):
    if cvss >= 9.0: return "P1"
    if cvss >= 7.0: return "P2"
    if cvss >= 4.0: return "P3"
    return "P4"

df["Baseline_tier"] = df["CVSS"].apply(cvss_to_tier)
df["Baseline_score"] = df["CVSS"]  # baseline ranks by raw CVSS

# Tier encoding: P1=1 (most urgent) ... P4=4 (least urgent)
TIER_TO_INT = {"P1": 1, "P2": 2, "P3": 3, "P4": 4}
TIER_ORDER = ["P1", "P2", "P3", "P4"]

# All methods to evaluate
# Dictionary to match each method name to its tier and score column
METHODS = {
    "CVSS-only":  {"tier": "Baseline_tier",  "score": "Baseline_score"},
    "DeepSeek":   {"tier": "DeepSeek_tier",  "score": "DeepSeek_score"},
    "GPT-5":      {"tier": "GPT5_tier",      "score": "GPT5_score"},
    "Claude":     {"tier": "Claude_tier",     "score": "Claude_score"},
}

print(f"Loaded {len(df)} vulnerabilities\n")

In [ ]:
df

### RQ1 - LLM Agreement

Same vulnerability is provided to three different LLMs, and we wish to compute the agreement between them.

We compute:
- Fleiss' kappa on tier
- Spearman correlation on continuous score

In [ ]:
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from scipy.stats import spearmanr

# Load master table
df = pd.read_csv("datasets/extracts/master_results.csv")

# Fleiss' kappa: needs a matrix of [n_items x n_categories] counts
tier_cols = ["DeepSeek_tier", "GPT5_tier", "Claude_tier"]
ratings, _ = aggregate_raters(df[tier_cols].values)
kappa = fleiss_kappa(ratings)
print(f"Fleiss kappa across 3 LLMs: {kappa:.3f}")

# Spearman on scores, pairwise
for a, b in [("DeepSeek_score","GPT5_score"), ("DeepSeek_score","Claude_score"), ("GPT5_score","Claude_score")]:
    rho, p = spearmanr(df[a], df[b])
    print(f"Spearman {a} vs {b}: ρ={rho:.3f}")

### RQ2 - LLM Agreement Against Experts

We attempt to determine how well the AI models have assigned priorities to vulnerabiliteis compared to expert labelled ground truth

**Classification:**

Did each individual model prediction matches against the ground truth? We use classification style analysis to answer this question.

- **A1. Weighted Cohen's Kappa**
- **A2. Per Tier F1, precision, recall**
- **A3. Macro-averaged F1**
- **A4. Confusion Matrix**
- **A5. Mean Absolute Error**

**A1. Weighted Cohens Kapps:**

Meausres the agreement between the models tier assigment and expert labels. Weighted because disagreement are worse depending on how off they are (P1 and P2 vs. P1 and P4)

In [ ]:
from sklearn.metrics import cohen_kappa_score

print("\n--- A1: Weighted Cohen's Kappa (linear) ---")
print(f"{'Method':<15} {'κ':>8} {'95% CI':>20}")
print("-" * 45)

def compute_weighted_kappa(y_true_series, y_pred_series):
    """Compute linearly-weighted Cohen's kappa."""
    return cohen_kappa_score(
        y_true_series, y_pred_series, 
        weights="linear",
        labels=TIER_ORDER,
    )

for name, cols in METHODS.items():
    y_pred = df[cols["tier"]]
    kappa = compute_weighted_kappa(y_true, y_pred)
    
    # Bootstrap 95% CI
    boot_kappas = []
    rng = np.random.default_rng(42)
    for _ in range(1000):
        idx = rng.integers(0, len(df), size=len(df))
        boot_kappas.append(compute_weighted_kappa(y_true.iloc[idx], y_pred.iloc[idx]))
    ci_low, ci_high = np.percentile(boot_kappas, [2.5, 97.5])
    
    print(f"{name:<15} {kappa:>8.3f} [{ci_low:.3f}, {ci_high:.3f}]")

**A2. Per Tier F1, precision, recall**

Determines how well each model performs on _individual tiers_, not just overall. Is the model better at predicting P1 then P3?

In [ ]:
from sklearn.metrics import classification_report

print("\n--- A2: Per-tier Precision / Recall / F1 ---")
for name, cols in METHODS.items():
    y_pred = df[cols["tier"]]
    print(f"\n  {name}:")
    report = classification_report(
        y_true, y_pred,
        labels=TIER_ORDER,
        target_names=TIER_ORDER,
        zero_division=0,
        output_dict=False,
    )
    # Indent the report for readability
    for line in report.split("\n"):
        print(f"    {line}")

**A3. Macro Averaged F1:**

Treats all tiers equally despite of how many fall into certain stratified cells. Ensures that performance on lower disagreement prone instances are just as important as performance on common instances 

In [ ]:
print("\n--- A3: Macro F1 Summary ---")
print(f"{'Method':<15} {'Macro F1':>10}")
print("-" * 27)
for name, cols in METHODS.items():
    y_pred = df[cols["tier"]]
    report_dict = classification_report(
        y_true, y_pred,
        labels=TIER_ORDER,
        output_dict=True,
        zero_division=0,
    )
    print(f"{name:<15} {report_dict['macro avg']['f1-score']:>10.3f}")

**A4. Confusion Matrix:**

Visualize what each model predicted vs what the experts predicted, plotted as a heatmap

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

print("\n--- A4: Confusion Matrices (saved as PNG) ---")
fig, axes = plt.subplots(1, len(METHODS), figsize=(5 * len(METHODS), 5))
for ax, (name, cols) in zip(axes, METHODS.items()):
    y_pred = df[cols["tier"]]
    cm = confusion_matrix(y_true, y_pred, labels=TIER_ORDER)
    disp = ConfusionMatrixDisplay(cm, display_labels=TIER_ORDER)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(name)
    ax.set_xlabel("Predicted Tier")
    ax.set_ylabel("Expert Consensus Tier")
plt.tight_layout()
plt.savefig("rq2_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved: rq2_confusion_matrices.png")

**A5. Mean Absolute Error:**

Measures how many number of tiers each prediction is off comapred to expert labells.

In [ ]:
print("\n--- A5: Mean Absolute Error (tier distance) ---")
print(f"{'Method':<15} {'MAE':>8} {'95% CI':>20}")
print("-" * 45)

y_true_int = y_true.map(TIER_TO_INT)

for name, cols in METHODS.items():
    y_pred_int = df[cols["tier"]].map(TIER_TO_INT)
    mae = np.abs(y_true_int - y_pred_int).mean()
    
    # Bootstrap CI
    boot_maes = []
    rng = np.random.default_rng(42)
    for _ in range(1000):
        idx = rng.integers(0, len(df), size=len(df))
        boot_maes.append(np.abs(y_true_int.iloc[idx] - y_pred_int.iloc[idx]).mean())
    ci_low, ci_high = np.percentile(boot_maes, [2.5, 97.5])
    
    print(f"{name:<15} {mae:>8.3f} [{ci_low:.3f}, {ci_high:.3f}]")

**Ranking:**

The idea is to determine if the model put the "right things" at the top. From an analysts perspective going through a backlog of vulnerabilities, this determines if he will find the more critical stuff early.

This approach relies more on the continious risk scores of the model outputs as they provide a finer grained understanding of the information as compared to labelled tiers

- **B1. Recall@K**
- **B2. Nomralized nDCG@K**
- **B3: Spearmen Rank Corelation**

**B1. Recall@K:**

From the top K% of model ranked vulnerabilities, how many are truly urgent vulnerabilities (P1+P2)

In [ ]:
def recall_at_k(scores, true_tiers, k_pct, target_tiers={"P1", "P2"}):
    """
    Rank vulns by score (descending = most urgent first).
    Take the top k_pct fraction.
    What fraction of the truly-urgent vulns are in that top slice?
    """
    n = len(scores)
    k = max(1, int(round(k_pct * n)))
    
    # Pair each vuln's score with its true tier, sort by score descending
    ranked = sorted(zip(scores, true_tiers), key=lambda x: -x[0])
    top_k_tiers = [tier for _, tier in ranked[:k]]
    
    # Count
    relevant_in_top_k = sum(1 for t in top_k_tiers if t in target_tiers)
    total_relevant = sum(1 for t in true_tiers if t in target_tiers)
    
    if total_relevant == 0:
        return 0.0
    return relevant_in_top_k / total_relevant

print("\n--- B1: Recall@K (P1+P2 recovery) ---")
print(f"{'Method':<15} {'R@10%':>8} {'R@20%':>8} {'R@30%':>8}")
print("-" * 42)

for name, cols in METHODS.items():
    scores = df[cols["score"]].values
    true_tiers = df["Consensus_tier"].values
    
    r10 = recall_at_k(scores, true_tiers, 0.10)
    r20 = recall_at_k(scores, true_tiers, 0.20)
    r30 = recall_at_k(scores, true_tiers, 0.30)
    
    print(f"{name:<15} {r10:>8.1%} {r20:>8.1%} {r30:>8.1%}")

Rerun the recall@K with bootstrap CIs and report the delta versus the CVSS baseline

In [ ]:
print("\n--- B1b: Recall@20% with Bootstrap CI ---")
print(f"{'Method':<15} {'R@20%':>8} {'95% CI':>20} {'Δ vs CVSS':>12}")
print("-" * 58)

baseline_r20 = recall_at_k(
    df["Baseline_score"].values, df["Consensus_tier"].values, 0.20
)

for name, cols in METHODS.items():
    scores = df[cols["score"]].values
    true_tiers = df["Consensus_tier"].values
    
    r20 = recall_at_k(scores, true_tiers, 0.20)
    
    # Bootstrap CI
    boot_r20s = []
    rng = np.random.default_rng(42)
    for _ in range(1000):
        idx = rng.integers(0, len(df), size=len(df))
        boot_r20s.append(recall_at_k(scores[idx], true_tiers[idx], 0.20))
    ci_low, ci_high = np.percentile(boot_r20s, [2.5, 97.5])
    
    delta = r20 - baseline_r20 if name != "CVSS-only" else 0.0
    delta_str = f"+{delta:.1%}" if delta > 0 else f"{delta:.1%}" if name != "CVSS-only" else "—"
    
    print(f"{name:<15} {r20:>8.1%} [{ci_low:.1%}, {ci_high:.1%}] {delta_str:>12}")

**B2. Normalized nDCG:**

Determines how good the models rankings are compared to ideal ranking

In [ ]:
from sklearn.metrics import ndcg_score

TIER_TO_RELEVANCE = {"P1": 3, "P2": 2, "P3": 1, "P4": 0}

def compute_ndcg_at_k(scores, true_tiers, k_pct):
    """
    Compute nDCG@K.
    scores: model's continuous scores (higher = more urgent)
    true_tiers: expert consensus tiers
    k_pct: fraction of the list to evaluate (e.g., 0.2 for top 20%)
    """
    n = len(scores)
    k = max(1, int(round(k_pct * n)))
    
    # Convert tiers to relevance scores
    relevances = np.array([TIER_TO_RELEVANCE[t] for t in true_tiers])
    
    # sklearn's ndcg_score expects 2D arrays:
    #   y_true = [relevance values]  (shape 1 x n)
    #   y_score = [model scores]     (shape 1 x n)
    # It internally sorts by y_score and computes nDCG.
    
    return ndcg_score(
        y_true=[relevances],
        y_score=[scores],
        k=k,
    )

print("\n--- B2: nDCG@K ---")
print(f"{'Method':<15} {'nDCG@10%':>10} {'nDCG@20%':>10} {'nDCG@30%':>10}")
print("-" * 48)

for name, cols in METHODS.items():
    scores = df[cols["score"]].values
    true_tiers = df["Consensus_tier"].values
    
    ndcg10 = compute_ndcg_at_k(scores, true_tiers, 0.10)
    ndcg20 = compute_ndcg_at_k(scores, true_tiers, 0.20)
    ndcg30 = compute_ndcg_at_k(scores, true_tiers, 0.30)
    
    print(f"{name:<15} {ndcg10:>10.3f} {ndcg20:>10.3f} {ndcg30:>10.3f}")

Apply same bootstrap logic

In [ ]:
print("\n--- B2b: nDCG@20% with Bootstrap CI ---")
print(f"{'Method':<15} {'nDCG@20%':>10} {'95% CI':>20}")
print("-" * 48)

for name, cols in METHODS.items():
    scores = df[cols["score"]].values
    true_tiers = df["Consensus_tier"].values
    
    ndcg20 = compute_ndcg_at_k(scores, true_tiers, 0.20)
    
    boot_ndcgs = []
    rng = np.random.default_rng(42)
    for _ in range(1000):
        idx = rng.integers(0, len(df), size=len(df))
        boot_ndcgs.append(compute_ndcg_at_k(scores[idx], true_tiers[idx], 0.20))
    ci_low, ci_high = np.percentile(boot_ndcgs, [2.5, 97.5])
    
    print(f"{name:<15} {ndcg20:>10.3f} [{ci_low:.3f}, {ci_high:.3f}]")

**B3. Spearmen Rank Corelation:**

Measures if the models continious risk score produces the same ordering of the vulnerabilities as the expert tiers.

In [ ]:
print("\n--- B3: Spearman ρ (model score vs expert tier-as-rank) ---")
print(f"{'Method':<15} {'ρ':>8} {'p-value':>12}")
print("-" * 38)

# Convert consensus tier to numeric (P1=1 is most urgent, lower = more urgent)
# For Spearman we want higher score = higher urgency, so invert: P1=4, P4=1
TIER_TO_RANK = {"P1": 4, "P2": 3, "P3": 2, "P4": 1}
expert_ranks = df["Consensus_tier"].map(TIER_TO_RANK)

for name, cols in METHODS.items():
    scores = df[cols["score"]]
    rho, pval = spearmanr(scores, expert_ranks)
    print(f"{name:<15} {rho:>8.3f} {pval:>12.4f}")

### Cieling Comparison

Measures how the models compare to the inter-expert agreement. The expert agreement is treated as the cieling, as such a model should not be expected to beat it.

- **C1. Inter-expert agreement**
- **C2. Model vs Consensus**
- **C3. Expert vs Consensus (sanity check)**

**C1. Inter-expert Agreement:**

The agreeemnt is calculated by comparing each expert with the consensus and with each other expert. Weighted kappa is used as the metric of choice

In [ ]:
print("\n--- C1: Inter-expert Agreement (the ceiling) ---")

expert_cols = ["Expert1_tier", "Expert2_tier"]
# Add Expert3_tier if you have 3 experts:
# expert_cols = ["Expert1_tier", "Expert2_tier", "Expert3_tier"]

print("\n  Pairwise expert κ:")
expert_kappas = []
for i in range(len(expert_cols)):
    for j in range(i + 1, len(expert_cols)):
        k = compute_weighted_kappa(df[expert_cols[i]], df[expert_cols[j]])
        expert_kappas.append(k)
        print(f"    {expert_cols[i]} vs {expert_cols[j]}: κ = {k:.3f}")

mean_expert_kappa = np.mean(expert_kappas)
print(f"\n  Mean inter-expert κ: {mean_expert_kappa:.3f}")

Bootstrap the mean inter expert kappa

In [ ]:
# Bootstrap CI on mean inter-expert kappa
boot_expert_kappas = []
rng = np.random.default_rng(42)
for _ in range(1000):
    idx = rng.integers(0, len(df), size=len(df))
    pairwise = []
    for i in range(len(expert_cols)):
        for j in range(i + 1, len(expert_cols)):
            pairwise.append(
                compute_weighted_kappa(df[expert_cols[i]].iloc[idx], df[expert_cols[j]].iloc[idx])
            )
    boot_expert_kappas.append(np.mean(pairwise))
ci_low, ci_high = np.percentile(boot_expert_kappas, [2.5, 97.5])
print(f"  95% CI: [{ci_low:.3f}, {ci_high:.3f}]")


**C2. Model vs Consensus:**

For each method, the kappa of model vs consensus should be within the range of the inter-expert agreement.

In [ ]:
print("\n--- C2: Model vs Consensus, Compared to Expert Ceiling ---")
print(f"\n  Expert ceiling: κ = {mean_expert_kappa:.3f} [{ci_low:.3f}, {ci_high:.3f}]")
print(f"\n  {'Method':<15} {'κ vs consensus':>15} {'Within ceiling CI?':>20}")
print("  " + "-" * 52)

for name, cols in METHODS.items():
    y_pred = df[cols["tier"]]
    k = compute_weighted_kappa(y_true, y_pred)
    within = "YES ✓" if ci_low <= k <= ci_high else ("ABOVE ↑" if k > ci_high else "BELOW ↓")
    print(f"  {name:<15} {k:>15.3f} {within:>20}")

print("""
  Interpretation:
    YES ✓  = model is statistically indistinguishable from a human expert
    BELOW ↓ = model performs worse than human agreement allows
    ABOVE ↑ = model exceeds human agreement (unusual; check for overfitting)
""")


**C3. Expert vs Consenus (sanity check):**

Sanity check incase model outperforms the expert agreement. If there is such a case, something is wrong with the data. Sanity check should give high k values since the consensus was derived from the experts

In [ ]:
print("--- C3: Individual Expert vs Consensus (sanity check) ---")
for col in expert_cols:
    k = compute_weighted_kappa(y_true, df[col])
    print(f"  {col} vs Consensus: κ = {k:.3f}")
print("  (These should be high since consensus was derived from these experts)")